<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fa9200;">Identify Key Stats based on Opening Market Trends
    
<span style="font-family: Arial; font-weight:bold;font-size:2.1em;color:#fff;">-----------------------------------

# Import Packages

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import date, datetime, timedelta
import os
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
import statsmodels.formula.api as sm
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import yfinance as yf
from scipy.signal import argrelextrema
from collections import defaultdict
import sqlite3
import warnings
warnings.filterwarnings("ignore")
import Indicators
import Measurement
import Charts

# Grab Data for one Ticker

In [68]:
tickers = Indicators.save_sp500_tickers()
ticker = tickers[0]
ticker = 'GOOG'
#ticker = 'WDC'
events = {'ihs_event':'bull','hs_event':'bear','fw_event':'bull','rw_event':'bear'}
df = Indicators.get_ticker(ticker,500)
df['day'] = df['date'].dt.date
df

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker,day
0,2022-07-28 09:30:00-04:00,113.250000,113.949997,111.850998,112.379997,6034017,0.0,0.0,2022-07-28 09:30:00,GOOG,2022-07-28
1,2022-07-28 10:30:00-04:00,112.360001,114.070000,111.950104,113.989998,3628023,0.0,0.0,2022-07-28 10:30:00,GOOG,2022-07-28
2,2022-07-28 11:30:00-04:00,113.989998,114.279999,113.339996,113.754601,2859153,0.0,0.0,2022-07-28 11:30:00,GOOG,2022-07-28
3,2022-07-28 12:30:00-04:00,113.754997,113.800003,113.180000,113.750000,1930649,0.0,0.0,2022-07-28 12:30:00,GOOG,2022-07-28
4,2022-07-28 13:30:00-04:00,113.750000,114.300102,113.684799,113.964996,1654304,0.0,0.0,2022-07-28 13:30:00,GOOG,2022-07-28
...,...,...,...,...,...,...,...,...,...,...,...
3483,2024-07-24 11:30:00-04:00,175.619995,176.039993,174.070007,174.250000,3448525,0.0,0.0,2024-07-24 11:30:00,GOOG,2024-07-24
3484,2024-07-24 12:30:00-04:00,174.250000,175.509995,174.220001,175.015396,2147494,0.0,0.0,2024-07-24 12:30:00,GOOG,2024-07-24
3485,2024-07-24 13:30:00-04:00,175.000000,175.604996,174.089996,174.389999,2504787,0.0,0.0,2024-07-24 13:30:00,GOOG,2024-07-24
3486,2024-07-24 14:30:00-04:00,174.419998,174.669998,173.880005,174.320007,2511479,0.0,0.0,2024-07-24 14:30:00,GOOG,2024-07-24


# Add SMA, Min/Max, Events

In [63]:
SMAs = [5,30,60,90]
smoothing = 7
window = 7

df = Indicators.get_sma(df,SMAs)
events_df = {}
minmax = Indicators.get_max_min(df, smoothing, window)
for event in events:
    events_df[event] = getattr(Indicators,event)(minmax)
df

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker,day,SMA5,SMA30,SMA60,SMA90
0,2022-07-28 09:30:00-04:00,113.250000,113.949997,111.850998,112.379997,6034017,0.0,0.0,2022-07-28 09:30:00,GOOG,2022-07-28,NaN,NaN,NaN,NaN
1,2022-07-28 10:30:00-04:00,112.360001,114.070000,111.950104,113.989998,3628023,0.0,0.0,2022-07-28 10:30:00,GOOG,2022-07-28,NaN,NaN,NaN,NaN
2,2022-07-28 11:30:00-04:00,113.989998,114.279999,113.339996,113.754601,2859153,0.0,0.0,2022-07-28 11:30:00,GOOG,2022-07-28,NaN,NaN,NaN,NaN
3,2022-07-28 12:30:00-04:00,113.754997,113.800003,113.180000,113.750000,1930649,0.0,0.0,2022-07-28 12:30:00,GOOG,2022-07-28,NaN,NaN,NaN,NaN
4,2022-07-28 13:30:00-04:00,113.750000,114.300102,113.684799,113.964996,1654304,0.0,0.0,2022-07-28 13:30:00,GOOG,2022-07-28,113.567918,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3483,2024-07-24 11:30:00-04:00,175.619995,176.039993,174.070007,174.250000,3448525,0.0,0.0,2024-07-24 11:30:00,GOOG,2024-07-24,178.996582,181.152948,183.628661,185.993472
3484,2024-07-24 12:30:00-04:00,174.250000,175.509995,174.220001,175.015396,2147494,0.0,0.0,2024-07-24 12:30:00,GOOG,2024-07-24,177.058661,181.009795,183.424084,185.803533
3485,2024-07-24 13:30:00-04:00,175.000000,175.604996,174.089996,174.389999,2504787,0.0,0.0,2024-07-24 13:30:00,GOOG,2024-07-24,175.218661,180.865461,183.216417,185.607200
3486,2024-07-24 14:30:00-04:00,174.419998,174.669998,173.880005,174.320007,2511479,0.0,0.0,2024-07-24 14:30:00,GOOG,2024-07-24,174.718661,180.704462,182.998834,185.411200


### Add Previous Close price, Opening Day price, and the high from the second hour of the day

In [64]:
#df['day_open'] = df.mask.groupby([
df['day_start'] = df.groupby(['day'])['date'].transform('min')
df1 = df[df['date']==df['day_start']].copy()
df1.reset_index(inplace=True,drop=True)
df['day_open'] = df['day_start'].replace(dict(zip(df1['date'],df1['open'])))
df['day_end'] = df.groupby(['day'])['date'].transform('max')
df1 = df[df['date']==df['day_end']].copy()
df1.reset_index(inplace=True,drop=True)
df1['prev_close'] = df1['close'].shift(1)
df['prev_close'] = df['day_end'].replace(dict(zip(df1['date'],df1['prev_close'])))
df['open_chng'] = (df['day_open'] - df['prev_close'])/df['day_open']
df['open_chng'] = pd.to_numeric(df['open_chng'], errors='coerce')
df['next_high'] = df['high'].shift(-1)
df['next_high_chng'] = (df['next_high'] - df['day_open'])/df['day_open']
df['next_high_chng'] = pd.to_numeric(df['next_high_chng'], errors='coerce')
df.head(10)

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker,...,SMA30,SMA60,SMA90,day_start,day_open,day_end,prev_close,open_chng,next_high,next_high_chng
0,2022-07-28 09:30:00-04:00,113.250000,113.949997,111.850998,112.379997,6034017,0.0,0.0,2022-07-28 09:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.070000,0.007241
1,2022-07-28 10:30:00-04:00,112.360001,114.070000,111.950104,113.989998,3628023,0.0,0.0,2022-07-28 10:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.279999,0.009095
2,2022-07-28 11:30:00-04:00,113.989998,114.279999,113.339996,113.754601,2859153,0.0,0.0,2022-07-28 11:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,113.800003,0.004857
3,2022-07-28 12:30:00-04:00,113.754997,113.800003,113.180000,113.750000,1930649,0.0,0.0,2022-07-28 12:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.300102,0.009272
4,2022-07-28 13:30:00-04:00,113.750000,114.300102,113.684799,113.964996,1654304,0.0,0.0,2022-07-28 13:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.650002,0.012362
5,2022-07-28 14:30:00-04:00,113.959999,114.650002,113.910004,114.540001,2020668,0.0,0.0,2022-07-28 14:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.699997,0.012804
6,2022-07-28 15:30:00-04:00,114.540001,114.699997,114.279999,114.589996,2162452,0.0,0.0,2022-07-28 15:30:00,GOOG,...,NaN,NaN,NaN,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,115.559998,0.020397
7,2022-07-29 09:30:00-04:00,113.639999,115.559998,113.470001,115.430000,8027052,0.0,0.0,2022-07-29 09:30:00,GOOG,...,NaN,NaN,NaN,2022-07-29 09:30:00,113.639999,2022-07-29 15:30:00,114.589996,-0.00836,115.575401,0.017031
8,2022-07-29 10:30:00-04:00,115.449997,115.575401,114.019997,114.180000,4205053,0.0,0.0,2022-07-29 10:30:00,GOOG,...,NaN,NaN,NaN,2022-07-29 09:30:00,113.639999,2022-07-29 15:30:00,114.589996,-0.00836,115.129997,0.013112
9,2022-07-29 11:30:00-04:00,114.180000,115.129997,114.080002,114.879997,1999212,0.0,0.0,2022-07-29 11:30:00,GOOG,...,NaN,NaN,NaN,2022-07-29 09:30:00,113.639999,2022-07-29 15:30:00,114.589996,-0.00836,115.641602,0.017614


### Keep only first hour of the day and Create Buckets

In [65]:
df2 = df[df['date']==df['day_start']].copy()
# First, we need to calculate our IQR.
field_name = 'open_chng'
q1 = df2[field_name].quantile(0.25)                 
q3 = df2[field_name].quantile(0.75)
iqr = q3 - q1

# Now let's calculate upper and lower bounds.
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr

# Let us create our bins:
num_bins = 10
bin_width = (upper - lower) / num_bins
bins = [lower + i*bin_width for i in range(num_bins)]
bins += [upper, float('inf')]  # Now we add our last bin, which will contain any value greater than the upper-bound of the IQR.

format(1/3, ".0%")
# Let us create our labels:
labels = [f'Bucket {format(bins[i], ".2%")} to {format(bins[i+1], ".2%")}' for i in range(1,num_bins+1)]
labels.append('Outside IQR')

# Finally, we add a new column to the df:
df2[field_name + '_bucket'] = pd.cut(df2[field_name], bins=bins, labels=labels)
df2.open_chng.describe()

count    499.000000
mean      -0.000626
std        0.013485
min       -0.084642
25%       -0.005306
50%       -0.000210
75%        0.004728
max        0.102519
Name: open_chng, dtype: float64

### Run Summary Stats

In [66]:
#df2[(df2.open_chng>=0.01)&(df2.open_chng<=0.02)].next_high_chng.describe()
df2.groupby(field_name + '_bucket').next_high_chng.describe()

,count,mean,std,min,25%,50%,75%,max
open_chng_bucket,,,,,,,,
Bucket -1.63% to -1.23%,18.0,0.003892,0.008707,-0.011653,-0.004096,0.004662,0.011531,0.015781
Bucket -1.23% to -0.83%,19.0,0.000232,0.011089,-0.015408,-0.009163,-0.000770,0.006828,0.024492
Bucket -0.83% to -0.43%,28.0,0.005191,0.008981,-0.011985,-0.000789,0.004083,0.010685,0.026407
Bucket -0.43% to -0.03%,77.0,0.004900,0.008210,-0.010194,-0.000816,0.004770,0.009515,0.027391
Bucket -0.03% to 0.37%,91.0,0.005368,0.009674,-0.019041,-0.000946,0.005493,0.011357,0.033021
Bucket 0.37% to 0.77%,106.0,0.006143,0.009429,-0.013123,0.000047,0.005213,0.010899,0.041049
Bucket 0.77% to 1.18%,64.0,0.003203,0.010837,-0.043465,-0.002491,0.002609,0.010817,0.022778
Bucket 1.18% to 1.58%,46.0,0.006092,0.010698,-0.019667,-0.000083,0.006238,0.011660,0.029459
Bucket 1.58% to 1.98%,12.0,0.003235,0.014083,-0.023294,-0.005389,0.001287,0.013418,0.028221


### Create Functions From Above

In [67]:
def add_prices(df):
    df['day_start'] = df.groupby(['day'])['date'].transform('min')
    df1 = df[df['date']==df['day_start']].copy()
    df1.reset_index(inplace=True,drop=True)
    df['day_open'] = df['day_start'].replace(dict(zip(df1['date'],df1['open'])))
    df['day_end'] = df.groupby(['day'])['date'].transform('max')
    df1 = df[df['date']==df['day_end']].copy()
    df1.reset_index(inplace=True,drop=True)
    df1['prev_close'] = df1['close'].shift(1)
    df['prev_close'] = df['day_end'].replace(dict(zip(df1['date'],df1['prev_close'])))
    df['open_chng'] = (df['day_open'] - df['prev_close'])/df['day_open']
    df['open_chng'] = pd.to_numeric(df['open_chng'], errors='coerce')
    df['next_high'] = df['high'].shift(-1)
    df['next_high_chng'] = (df['next_high'] - df['day_open'])/df['day_open']
    df['next_high_chng'] = pd.to_numeric(df['next_high_chng'], errors='coerce')
    
    return(df)

In [69]:
test = add_prices(df)
test

,datetime,open,high,low,close,volume,dividends,stock splits,date,ticker,day,day_start,day_open,day_end,prev_close,open_chng,next_high,next_high_chng
0,2022-07-28 09:30:00-04:00,113.250000,113.949997,111.850998,112.379997,6034017,0.0,0.0,2022-07-28 09:30:00,GOOG,2022-07-28,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.070000,0.007241
1,2022-07-28 10:30:00-04:00,112.360001,114.070000,111.950104,113.989998,3628023,0.0,0.0,2022-07-28 10:30:00,GOOG,2022-07-28,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.279999,0.009095
2,2022-07-28 11:30:00-04:00,113.989998,114.279999,113.339996,113.754601,2859153,0.0,0.0,2022-07-28 11:30:00,GOOG,2022-07-28,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,113.800003,0.004857
3,2022-07-28 12:30:00-04:00,113.754997,113.800003,113.180000,113.750000,1930649,0.0,0.0,2022-07-28 12:30:00,GOOG,2022-07-28,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.300102,0.009272
4,2022-07-28 13:30:00-04:00,113.750000,114.300102,113.684799,113.964996,1654304,0.0,0.0,2022-07-28 13:30:00,GOOG,2022-07-28,2022-07-28 09:30:00,113.250000,2022-07-28 15:30:00,NaT,NaN,114.650002,0.012362
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3483,2024-07-24 11:30:00-04:00,175.619995,176.039993,174.070007,174.250000,3448525,0.0,0.0,2024-07-24 11:30:00,GOOG,2024-07-24,2024-07-24 09:30:00,175.339996,2024-07-24 15:30:00,183.589996,-0.047051,175.509995,0.000970
3484,2024-07-24 12:30:00-04:00,174.250000,175.509995,174.220001,175.015396,2147494,0.0,0.0,2024-07-24 12:30:00,GOOG,2024-07-24,2024-07-24 09:30:00,175.339996,2024-07-24 15:30:00,183.589996,-0.047051,175.604996,0.001511
3485,2024-07-24 13:30:00-04:00,175.000000,175.604996,174.089996,174.389999,2504787,0.0,0.0,2024-07-24 13:30:00,GOOG,2024-07-24,2024-07-24 09:30:00,175.339996,2024-07-24 15:30:00,183.589996,-0.047051,174.669998,-0.003821
3486,2024-07-24 14:30:00-04:00,174.419998,174.669998,173.880005,174.320007,2511479,0.0,0.0,2024-07-24 14:30:00,GOOG,2024-07-24,2024-07-24 09:30:00,175.339996,2024-07-24 15:30:00,183.589996,-0.047051,175.130005,-0.001198


In [75]:
def add_buckets(df2,field_name):
    # First, we need to calculate our IQR.
    q1 = df2[field_name].quantile(0.25)                 
    q3 = df2[field_name].quantile(0.75)
    iqr = q3 - q1

    # Now let's calculate upper and lower bounds.
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    # Let us create our bins:
    num_bins = 10
    bin_width = (upper - lower) / num_bins
    bins = [lower + i*bin_width for i in range(num_bins)]
    bins += [upper, float('inf')]  # Now we add our last bin, which will contain any value greater than the upper-bound of the IQR.

    # Let us create our labels:
    labels = [f'Bucket {format(bins[i], ".2%")} to {format(bins[i+1], ".2%")}' for i in range(1,num_bins+1)]
    labels.append('Outside IQR')

    # Finally, we add a new column to the df:
    df2[field_name + '_bucket'] = pd.cut(df2[field_name], bins=bins, labels=labels)
    
    return(df2)

In [72]:
field_name = 'open_chng'
test2 = test[test['date']==test['day_start']].copy()
test2 = add_buckets(test2)
test2.groupby(field_name + '_bucket').next_high_chng.describe()

,count,mean,std,min,25%,50%,75%,max
open_chng_bucket,,,,,,,,
Bucket -1.63% to -1.23%,18.0,0.003892,0.008707,-0.011653,-0.004096,0.004662,0.011531,0.015781
Bucket -1.23% to -0.83%,19.0,0.000232,0.011089,-0.015408,-0.009163,-0.000770,0.006828,0.024492
Bucket -0.83% to -0.43%,28.0,0.005191,0.008981,-0.011985,-0.000789,0.004083,0.010685,0.026407
Bucket -0.43% to -0.03%,77.0,0.004900,0.008210,-0.010194,-0.000816,0.004770,0.009515,0.027391
Bucket -0.03% to 0.37%,91.0,0.005368,0.009674,-0.019041,-0.000946,0.005493,0.011357,0.033021
Bucket 0.37% to 0.77%,106.0,0.006143,0.009429,-0.013123,0.000047,0.005213,0.010899,0.041049
Bucket 0.77% to 1.18%,64.0,0.003203,0.010837,-0.043465,-0.002491,0.002609,0.010817,0.022778
Bucket 1.18% to 1.58%,46.0,0.006092,0.010698,-0.019667,-0.000083,0.006238,0.011660,0.029459
Bucket 1.58% to 1.98%,12.0,0.003235,0.014083,-0.023294,-0.005389,0.001287,0.013418,0.028221


In [78]:
tickers = ['AMZN','GOOG','META','NFLX','T','EBAY','NRG','TSLA']
field_name = 'open_chng'
final = pd.DataFrame()
for ticker in tickers:
    df = Indicators.get_ticker(ticker,500)
    df['day'] = df['date'].dt.date
    df = add_prices(df)
    df2 = df[df['date']==df['day_start']].copy()
    final = pd.concat([final,df2],axis=0)

final = add_buckets(final,field_name)
final.groupby(field_name + '_bucket').next_high_chng.describe()

,count,mean,std,min,25%,50%,75%,max
open_chng_bucket,,,,,,,,
Bucket -1.74% to -1.30%,89.0,0.006406,0.017490,-0.033780,-0.004285,0.005633,0.013182,0.061155
Bucket -1.30% to -0.87%,149.0,0.008362,0.016935,-0.020793,-0.003919,0.005408,0.018105,0.071436
Bucket -0.87% to -0.44%,267.0,0.005945,0.013596,-0.038718,-0.002239,0.005422,0.014950,0.043172
Bucket -0.44% to -0.01%,523.0,0.005971,0.011736,-0.036091,-0.001275,0.004907,0.012715,0.060400
Bucket -0.01% to 0.43%,826.0,0.005108,0.011485,-0.036187,-0.002257,0.004715,0.011802,0.056696
Bucket 0.43% to 0.86%,861.0,0.005480,0.011689,-0.055975,-0.001413,0.004430,0.011987,0.074240
Bucket 0.86% to 1.29%,514.0,0.005274,0.011380,-0.043465,-0.001966,0.004750,0.012176,0.052541
Bucket 1.29% to 1.72%,284.0,0.004420,0.013582,-0.040489,-0.004878,0.003665,0.012125,0.048363
Bucket 1.72% to 2.16%,132.0,0.005686,0.014904,-0.046319,-0.003197,0.005982,0.015281,0.052341
